# 01 - Run an LLM from Python (`pyneat.genai.GenAIModel`)

This notebook runs a generative model end to end: load a model directory, send a prompt, read the
answer - then grow that into a system prompt, chat history, and token streaming.

> Loading a 4B model takes a few minutes and several GB of memory. Run these cells on the DevKit.
> Each model-call cell is written as code you run yourself; an **Expected output** section lets you
> confirm the result.

Adapted from `/workspace/core/tutorials/019_run_an_llm/run_an_llm.py` and verified against the
`pyneat` bindings in `/workspace/core/python/src/module.cpp`.

## Direct model vs server - which to use

Two ways to call a GenAI model:

- **Direct `model.run(request)` / `model.stream(request)`** - lowest overhead, in-process. Best for
  embedded application logic, scripts, and tests. **This notebook uses the direct path.**
- **`GenAIServer` (HTTP, OpenAI-compatible)** - use when the boundary is a network: a browser UI, a
  companion service, or a remote client that should not link the Neat runtime. Covered in
  `05-genai-server/`.

Start direct. Reach for the server only when a network boundary actually buys you something.

## Which handle: `GenAIModel` vs `VisionLanguageModel`

Three handles exist (bindings: `module.cpp`, headers: `/workspace/core/include/genai/`):

| Handle | Use for |
| --- | --- |
| `pyneat.genai.GenAIModel` | **Auto-detects** LLM / VLM / ASR from the directory. Start here. |
| `pyneat.genai.VisionLanguageModel` | Text-only LLMs and image-capable VLMs (narrower API). |
| `pyneat.genai.ASRModel` | Speech-to-text models. |

For a text LLM we use `GenAIModel` - it auto-detects the task and exposes capability checks
(`accepts_text()`, `accepts_image()`, `accepts_audio()`, `task()`, `model_id()`).

## Memory implications - read before you load

- Loading the model makes its **weights resident** in memory. `Qwen3-4B-...-a16w4` is a 4B model at
  4-bit weights (`a16w4` = 16-bit activations / 4-bit weights).
- Each request builds a **KV cache** that grows with the number of tokens in the prompt + history +
  generated output. `max_new_tokens` caps the generated part.
- **Your application owns the chat history.** The model does not remember previous `run()` calls; you
  rebuild `request.messages` each turn. Longer history = larger KV cache = slower time-to-first-token.
  Keep only the history you need.
- Confirm the model is present with `llima list` before loading, and check free space with
  `df -h /` first - a GenAI model directory can be several GB and the DevKit root filesystem is small.

## Step 1 - Load the model

`GenAIModel(path)` loads the deployed LLiMa model directory, e.g.
`/media/nvme/llima/models/Qwen3-4B-Instruct-2507-GPTQ-a16w4`.

In [ ]:
# Loads model weights into memory.
import pyneat as neat

MODEL_DIR = "/media/nvme/llima/models/Qwen3-4B-Instruct-2507-GPTQ-a16w4"

model = neat.genai.GenAIModel(MODEL_DIR)
print("model_id     :", model.model_id())
print("task         :", model.task())          # GenAITask.VisionLanguage for text/vision LLMs
print("accepts_text :", model.accepts_text())
print("accepts_image:", model.accepts_image())
print("accepts_audio:", model.accepts_audio())

**Interpretation.** `task()` returns `GenAITask.VisionLanguage` for both text-only LLMs and VLMs -
they share the same task family; you distinguish them with `accepts_image()`. For this text model
expect `accepts_text=True`, `accepts_image=False`, `accepts_audio=False`. (Enum values from
`module.cpp`: `GenAITask.VisionLanguage`, `GenAITask.ASR`.)

## Step 2 - One prompt

The shortest path: set `request.prompt` and a token budget, call `run()`, print `result.text`.
`run()` is synchronous - it returns after generation finishes.

In [ ]:
request = neat.genai.GenerationRequest()
request.prompt = "Give me three practical tips for designing a small REST API."
request.max_new_tokens = 96

result = model.run(request)
print("assistant:", result.text)
print("tokens:", result.metrics.generated_tokens,
      "| ttft_s:", round(result.metrics.time_to_first_token_s, 3),
      "| tok/s:", round(result.metrics.tokens_per_second, 2))

**Interpretation.** `result` is a `GenerationResult` with `.text`, `.finish_reason`, and
`.metrics` (a `GenerationMetrics`: `generated_tokens`, `time_to_first_token_s`, `tokens_per_second`).
Field names come straight from the bindings. `finish_reason` tells you whether generation stopped
naturally or hit `max_new_tokens`.

## Step 3 - System prompt

A short system instruction steers behavior. For a one-shot prompt, attach it with
`request.system_prompt`.

In [ ]:
concise = neat.genai.GenerationRequest()
concise.system_prompt = "You are concise and practical."
concise.prompt = "Give me one rule of thumb for designing a small REST API."
concise.max_new_tokens = 64

print("assistant:", model.run(concise).text)

## Step 4 - Chat history with `messages`

For conversation, use `request.messages` (a list of `ChatMessage`) instead of `request.prompt`. Each
`ChatMessage` has `.role` (`"system"` / `"user"` / `"assistant"`) and `.content`. **You** append the
assistant's reply to the list so the next turn sees it - the model keeps no state between `run()`
calls.

In [ ]:
def user_msg(text):
    m = neat.genai.ChatMessage(); m.role = "user"; m.content = text; return m

def system_msg(text):
    m = neat.genai.ChatMessage(); m.role = "system"; m.content = text; return m

messages = [system_msg("You are concise and practical."),
            user_msg("Give me three practical tips for writing API documentation.")]

chat = neat.genai.GenerationRequest()
chat.messages = messages
chat.max_new_tokens = 96
answer = model.run(chat)
print("assistant:", answer.text)

# Persist the reply so the follow-up sees it (the app owns history):
reply = neat.genai.ChatMessage(); reply.role = "assistant"; reply.content = answer.text
messages.append(reply)

# Follow-up turn - the model now sees the whole conversation:
messages.append(user_msg("Which tip should I apply first for a prototype?"))
follow_up = neat.genai.GenerationRequest()
follow_up.messages = messages
follow_up.max_new_tokens = 96
print("assistant:", model.run(follow_up).text)

**Interpretation.** Note the pattern: build list -> run -> append assistant reply -> append next
user turn -> run again. If you forget to append the assistant reply, the model loses the thread. This
is the single most common LLM-app bug, and it is by design: history is the application's job.

## Step 5 - Stream tokens

For UI-style output, call `stream()` instead of `run()`. It returns a `GenerationStream` you iterate;
each item is a `TokenSample` with the latest `.text` fragment. Call `.cancel()` if the user aborts.

In [ ]:
messages.append(user_msg("Turn that advice into a short checklist."))
streaming = neat.genai.GenerationRequest()
streaming.messages = messages
streaming.max_new_tokens = 96

stream_handle = model.stream(streaming)
print("assistant: ", end="", flush=True)
for token in stream_handle:      # each token is a TokenSample
    print(token.text, end="", flush=True)
    if token.is_final:
        break
print()
# stream_handle.cancel()  # call this to abort early (e.g. user closed the request)

**Interpretation.** `stream()` gives you tokens as they are produced, so a UI can render
progressively and the perceived latency is the time-to-first-token, not the full generation time.
`TokenSample` carries `.text`, `.metrics`, `.is_final`, `.finish_reason` (from the bindings).

## Generation options - the knobs that matter

Set on `GenerationRequest` (fields verified in `module.cpp` / `GenAITypes.h`):

| Field | Meaning |
| --- | --- |
| `prompt` | Single-shot user text (mutually exclusive in practice with `messages`). |
| `system_prompt` | System instruction for a single-shot prompt. |
| `messages` | List of `ChatMessage` for multi-turn chat (carries its own system message). |
| `max_new_tokens` | Cap on generated tokens. Bounds latency and KV-cache growth. |
| `images` | VLM image inputs (see `02_run_vlm`). |
| `audio` / `audio_file` / `language` | ASR inputs (see `03_audio_input_asr`). |
| `tools` / `tool_choice` | Tool-calling (JSON), for function-calling models. |

The corresponding CLI smoke test (no code) is `llima run --mode cli <model-id>`.

## Run it as a script

The whole flow is packaged as `scripts/run_llm.py`. On the DevKit:

```bash
dk /workspace/demo-neat/llima/02-run-llm-vlm/scripts/run_llm.py \
  --model /media/nvme/llima/models/Qwen3-4B-Instruct-2507-GPTQ-a16w4 \
  --prompt "Give me three tips for designing a small REST API."
```

## Expected output

When you run the cells (or `scripts/run_llm.py`) on the DevKit, expect:

- **Load:** `model_id: Qwen3-4B-Instruct-2507-GPTQ-a16w4`, `task: GenAITask.VisionLanguage`,
  `accepts_text: True`, `accepts_image: False`, `accepts_audio: False`.
- **One prompt:** a short answer with three REST-API tips, plus a metrics line
  (`tokens`, `ttft_s`, `tok/s`). Exact wording varies by model sampling.
- **System prompt:** a single terse rule of thumb.
- **Chat history:** a three-tip documentation answer, then a follow-up that references "the first
  tip" - proving the model saw the earlier turn.
- **Streaming:** the checklist prints token by token rather than all at once.

The run produces a simple prompt answer, a system-prompted answer, a context-aware follow-up, and a
streamed final response.

## References

- `/workspace/core/tutorials/019_run_an_llm/run_an_llm.py` and `README.md`
- `/workspace/core/python/src/module.cpp` (binding names, field names)
- `/workspace/core/include/genai/GenAIModel.h`, `GenAITypes.h`
- `/workspace/core/docs/develop-apps/development-workflow/genai-model.mdx`